In [10]:
import math

def sigmoid(z):
    """Convert a linear score into a probability."""
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    e = math.exp(z)
    return e / (1 + e)


def train_logistic(X, y, learning_rate=0.3, epochs=600, l2=0.01):
    """Train binary logistic regression using gradient descent."""

    n = len(X)
    n_features = len(X[0])

    weights = [0.0] * n_features
    bias = 0.0
    losses = []

    # Store nonzero features to avoid unnecessary calculations.
    sparse_X = [
        [(j, value) for j, value in enumerate(row) if value != 0]
        for row in X
    ]

    for epoch in range(epochs):
        dw = [0.0] * n_features
        db = 0.0
        total_loss = 0.0

        for features, target in zip(sparse_X, y):

            # Linear prediction: z = w*x + b
            z = sum(weights[j] * value for j, value in features) + bias

            # Convert to win probability
            p = sigmoid(z)

            # Binary cross-entropy
            p_safe = max(1e-15, min(1 - 1e-15, p))
            total_loss -= (
                target * math.log(p_safe)
                + (1 - target) * math.log(1 - p_safe)
            )

            # Gradients
            error = p - target
            db += error

            for j, value in features:
                dw[j] += error * value

        # Record loss, including L2 penalty
        regularization = (l2 / 2) * sum(w * w for w in weights)
        losses.append(total_loss / n + regularization)

        # Gradient descent updates
        for j in range(n_features):
            gradient = dw[j] / n + l2 * weights[j]
            weights[j] -= learning_rate * gradient

        bias -= learning_rate * db / n

    return {
        "weights": weights,
        "bias": bias,
        "losses": losses
    }


def predict_probability(model, x):
    """Predict the probability that Blue wins."""
    z = sum(
        w * value
        for w, value in zip(model["weights"], x)
    ) + model["bias"]

    return sigmoid(z)


In [11]:
import sys
import importlib
from pathlib import Path

# Locate the assignment directory containing src/data_setup.py
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for base in (cwd, *cwd.parents)
        for path in (base, base / "assignments" / "01_brawl_stars")
        if (path / "src" / "data_setup.py").is_file()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Cannot find src/data_setup.py from {cwd}"
    )

# Add assignment directory to Python's import path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import and reload Stage 1
import src.data_setup as data_setup
importlib.reload(data_setup)

read_matches = data_setup.read_matches
make_folds = data_setup.make_folds
experiment_sets = data_setup.experiment_sets
DATA_DIR = data_setup.DATA_DIR

# Verify E5 is available
originals = read_matches(DATA_DIR / "raw_matches.csv")
train, test = make_folds(originals)[0]

print("Project root:", PROJECT_ROOT)
print("Available experiments:")
print(list(experiment_sets(train, test).keys()))



Project root: /Users/hammondutra/Developer/cs156-machine-learning/assignments/01_brawl_stars
Available experiments:
['E0_prior', 'E1_original_signed', 'E2_augmented_signed', 'E3_original_separate', 'E4_brawl_only', 'E5_brawl_original']


In [12]:

# ------------------------------------------------------------
# TRAIN ALL EXPERIMENTS (E0–E5)
# ------------------------------------------------------------

originals = read_matches(DATA_DIR / "raw_matches.csv")
folds = make_folds(originals, k=5, seed=42)

results = {}

for fold_number, (train, test) in enumerate(folds, start=1):

    experiments = experiment_sets(train, test)
    print(f"\n--- Fold {fold_number} ---")

    for name, data in experiments.items():

        X_train = data["X_train"]
        y_train = data["y_train"]
        X_test = data["X_test"]
        y_test = data["y_test"]

        if name == "E0_prior":
            prior = sum(y_train) / len(y_train)
            probabilities = [prior] * len(X_test)

        else:
            model = train_logistic(
                X_train,
                y_train,
                learning_rate=0.3,
                epochs=600,
                l2=0.01
            )

            probabilities = [
                predict_probability(model, x)
                for x in X_test
            ]

        if name not in results:
            results[name] = []

        for pair_id, actual, predicted in zip(
            data["test_pair_ids"], y_test, probabilities
        ):
            results[name].append({
                "pair_id": pair_id,
                "actual": actual,
                "probability": predicted
            })

        print(f"{name}: {len(X_test)} test matches")

print("\nTraining complete!")
print({name: len(preds) for name, preds in results.items()})



--- Fold 1 ---
E0_prior: 34 test matches
E1_original_signed: 34 test matches
E2_augmented_signed: 34 test matches
E3_original_separate: 34 test matches
E4_brawl_only: 19 test matches
E5_brawl_original: 19 test matches

--- Fold 2 ---
E0_prior: 33 test matches
E1_original_signed: 33 test matches
E2_augmented_signed: 33 test matches
E3_original_separate: 33 test matches
E4_brawl_only: 19 test matches
E5_brawl_original: 19 test matches

--- Fold 3 ---
E0_prior: 33 test matches
E1_original_signed: 33 test matches
E2_augmented_signed: 33 test matches
E3_original_separate: 33 test matches
E4_brawl_only: 19 test matches
E5_brawl_original: 19 test matches

--- Fold 4 ---
E0_prior: 31 test matches
E1_original_signed: 31 test matches
E2_augmented_signed: 31 test matches
E3_original_separate: 31 test matches
E4_brawl_only: 18 test matches
E5_brawl_original: 18 test matches

--- Fold 5 ---
E0_prior: 31 test matches
E1_original_signed: 31 test matches
E2_augmented_signed: 31 test matches
E3_origin

In [13]:
# ------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------

# Test 1: Can logistic regression learn a simple pattern?
toy_X = [[-2], [-1], [1], [2]]
toy_y = [0, 0, 1, 1]

toy_model = train_logistic(toy_X, toy_y, epochs=600)

assert predict_probability(toy_model, [-1]) < 0.5
assert predict_probability(toy_model, [1]) > 0.5
assert toy_model["losses"][-1] < toy_model["losses"][0]

print("Test 1 passed: Logistic regression learns correctly.")


# Test 2: Is every original match evaluated exactly once?
assert "results" in globals(), "Run the training cell first."

for name, predictions in results.items():

    # E4 and E5 evaluate only Brawl Ball matches.
    expected = 93 if name.startswith(("E4", "E5")) else 162

    assert len(predictions) == expected, (
        f"{name}: Expected {expected}, got {len(predictions)}"
    )

    assert len({p["pair_id"] for p in predictions}) == expected, (
        f"{name}: Duplicate or missing test matches"
    )

    print(f"{name}: {len(predictions)} unique test matches — OK")

print("\nBoth sanity checks passed!")


Test 1 passed: Logistic regression learns correctly.
E0_prior: 162 unique test matches — OK
E1_original_signed: 162 unique test matches — OK
E2_augmented_signed: 162 unique test matches — OK
E3_original_separate: 162 unique test matches — OK
E4_brawl_only: 93 unique test matches — OK
E5_brawl_original: 93 unique test matches — OK

Both sanity checks passed!


In [14]:
def evaluate(predictions):
    """Evaluate held-out predictions from one experiment."""

    n = len(predictions)
    tp = tn = fp = fn = 0
    logloss = 0
    brier = 0

    for row in predictions:
        y = row["actual"]
        p = row["probability"]
        predicted = int(p >= 0.5)

        if y == 1 and predicted == 1:
            tp += 1
        elif y == 0 and predicted == 0:
            tn += 1
        elif y == 0 and predicted == 1:
            fp += 1
        else:
            fn += 1

        # Log loss
        p_safe = max(1e-15, min(1 - 1e-15, p))
        logloss -= (
            y * math.log(p_safe)
            + (1 - y) * math.log(1 - p_safe)
        )

        # Brier score
        brier += (p - y) ** 2

    accuracy = (tp + tn) / n

    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    balanced_accuracy = (sensitivity + specificity) / 2

    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "log_loss": logloss / n,
        "brier": brier / n,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn
    }


# Evaluate every experiment on held-out observations
scores = {
    name: evaluate(predictions)
    for name, predictions in results.items()
}

print(f"{'Experiment':<26} {'Accuracy':>9} {'Bal.Acc':>9} {'Log Loss':>10} {'Brier':>9}")
print("-" * 70)

for name, score in scores.items():
    print(
        f"{name:<26} "
        f"{score['accuracy']:>9.3f} "
        f"{score['balanced_accuracy']:>9.3f} "
        f"{score['log_loss']:>10.3f} "
        f"{score['brier']:>9.3f}"
    )


Experiment                  Accuracy   Bal.Acc   Log Loss     Brier
----------------------------------------------------------------------
E0_prior                       0.747     0.500      0.566     0.189
E1_original_signed             0.722     0.516      0.576     0.192
E2_augmented_signed            0.611     0.554      0.638     0.224
E3_original_separate           0.747     0.532      0.584     0.196
E4_brawl_only                  0.613     0.571      0.664     0.236
E5_brawl_original              0.688     0.521      0.626     0.214


In [15]:
# ------------------------------------------------------------
# BRAWL BALL COMPARISON — SAME TEST MATCHES
# ------------------------------------------------------------

# Identify all real Brawl Ball matches evaluated by E4.
brawl_ids = {
    row["pair_id"]
    for row in results["E4_brawl_only"]
}

# Evaluate every experiment on the same Brawl Ball matches.
brawl_scores = {}

for name, predictions in results.items():

    subset = [
        row for row in predictions
        if row["pair_id"] in brawl_ids
    ]

    assert len(subset) == len(brawl_ids)

    brawl_scores[name] = evaluate(subset)

# Display results table
print("BRAWL BALL ONLY — SAME 93 TEST MATCHES\n")
print(
    f"{'Experiment':<26} "
    f"{'Accuracy':>9} "
    f"{'Bal.Acc':>9} "
    f"{'Log Loss':>10} "
    f"{'Brier':>9}"
)
print("-" * 70)

for name, score in brawl_scores.items():
    print(
        f"{name:<26} "
        f"{score['accuracy']:>9.3f} "
        f"{score['balanced_accuracy']:>9.3f} "
        f"{score['log_loss']:>10.3f} "
        f"{score['brier']:>9.3f}"
    )

# ------------------------------------------------------------
# CONTROLLED COMPARISONS
# ------------------------------------------------------------

comparisons = [
    (
        "E1_original_signed",
        "E5_brawl_original",
        "Specialization WITHOUT mirroring"
    ),
    (
        "E2_augmented_signed",
        "E4_brawl_only",
        "Specialization WITH mirroring"
    ),
    (
        "E4_brawl_only",
        "E5_brawl_original",
        "Removing mirroring from Brawl Ball"
    )
]

print("\nCONTROLLED COMPARISONS")
print("(Positive improvement = second model performs better)\n")

for model_a, model_b, description in comparisons:

    improvement = (
        brawl_scores[model_a]["log_loss"]
        - brawl_scores[model_b]["log_loss"]
    )

    print(f"{description}:")
    print(f"  {model_a} -> {model_b}")
    print(f"  Log-loss improvement: {improvement:+.4f}\n")


BRAWL BALL ONLY — SAME 93 TEST MATCHES

Experiment                  Accuracy   Bal.Acc   Log Loss     Brier
----------------------------------------------------------------------
E0_prior                       0.731     0.500      0.583     0.197
E1_original_signed             0.699     0.516      0.574     0.194
E2_augmented_signed            0.645     0.618      0.613     0.214
E3_original_separate           0.731     0.538      0.613     0.206
E4_brawl_only                  0.613     0.571      0.664     0.236
E5_brawl_original              0.688     0.521      0.626     0.214

CONTROLLED COMPARISONS
(Positive improvement = second model performs better)

Specialization WITHOUT mirroring:
  E1_original_signed -> E5_brawl_original
  Log-loss improvement: -0.0520

Specialization WITH mirroring:
  E2_augmented_signed -> E4_brawl_only
  Log-loss improvement: -0.0509

Removing mirroring from Brawl Ball:
  E4_brawl_only -> E5_brawl_original
  Log-loss improvement: +0.0377

